In [2]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.data.load_data import load_resumes, load_jobs
from src.data.preprocess import clean_text
from src.models.recommender import build_recommender, recommend_jobs

In [4]:
resumes = load_resumes()
jobs = load_jobs().sample(50000, random_state=42).reset_index(drop=True)
jobs['clean_text'] = jobs['description'].apply(clean_text)
print("✅ Jobs ready:", jobs.shape)
print("✅ Resumes ready:", resumes.shape)

✅ Total jobs loaded: 148718
✅ Jobs ready: (50000, 6)
✅ Resumes ready: (962, 2)


In [5]:
print("Building TF-IDF matrix... (may take 2-3 mins)")
vectorizer, job_matrix = build_recommender(jobs, text_col='clean_text')
print("✅ Recommender built!")
print("Job matrix shape:", job_matrix.shape)

Building TF-IDF matrix... (may take 2-3 mins)
✅ Recommender built!
Job matrix shape: (50000, 5000)


In [6]:
sample_resume = resumes['Resume'].iloc[0]
clean = clean_text(sample_resume)
actual_category = resumes['Category'].iloc[0]

print(f"Testing for category: {actual_category}")
print("-" * 50)

results = recommend_jobs(clean, jobs, vectorizer, job_matrix, top_n=10)
print(results.to_string())

Testing for category: Data Science
--------------------------------------------------
                                                   title                               company                      location                                                                                                                                                    skills  match_score
15110                                   Forensic Analyst                                 Other                 Delhi,Gurgaon   forensic audit| document review| fraud investigation| forensic| forensic investigations| due diligence| fraud analytics| data analysis| fraud detection        36.92
9586    Hiring Data Scientist Professionals in Hyderabad  System Design/Implementation/ERP/CRM                     Hyderabad                   R| Time Series Analysis| Text Analytics| SQL| Machine Learning| Natural Language Processing| Finance| Data Science| Python| Forecasting        35.26
1906          Data Scientist (Human Se

In [8]:
sample_resume2 = resumes['Resume'].iloc[100]
clean2 = clean_text(sample_resume2)
category2 = resumes['Category'].iloc[100]

print(f"Testing for category: {category2}")
print("-" * 50)
results2 = recommend_jobs(clean2, jobs, vectorizer, job_matrix, top_n=10)
print(results2[['title', 'company', 'match_score']].to_string())

Testing for category: Advocate
--------------------------------------------------
                                                                                   title                                        company  match_score
9529                                                                           Paralegal                                                       26.80
2459                                                   Open Rank Lecturer, Communication                         University of Kentucky        24.53
24044                                                                   Teaching Faculty                           Medical Professional        24.29
26251                                                        Graduate Programs Recruiter                       Anderson University (SC)        23.84
15882                                                                          Assistant                                                       23.26
1338                    

In [10]:
for idx in [0, 100, 200, 300]:
    resume_text = clean_text(resumes['Resume'].iloc[idx])
    cat = resumes['Category'].iloc[idx]
    res = recommend_jobs(resume_text, jobs, vectorizer, job_matrix, top_n=3)
    print(f"\n📄 Category: {cat}")
    print(res[['title', 'match_score']].to_string())
    print("-" * 50)


📄 Category: Data Science
                                                   title  match_score
15110                                   Forensic Analyst        36.92
9586    Hiring Data Scientist Professionals in Hyderabad        35.26
1906          Data Scientist (Human Services experience)        34.65
--------------------------------------------------

📄 Category: Advocate
                                   title  match_score
9529                           Paralegal        26.80
2459   Open Rank Lecturer, Communication        24.53
24044                   Teaching Faculty        24.29
--------------------------------------------------

📄 Category: Mechanical Engineer
                                 title  match_score
37157       Senior Mechanical Engineer        37.27
44976  Mechanical Engineering Designer        37.08
11881                   Sales Engineer        37.06
--------------------------------------------------

📄 Category: Civil Engineer
                                  

In [11]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
joblib.dump(job_matrix, '../models/job_matrix.pkl')
joblib.dump(jobs, '../models/jobs_df.pkl')
print("✅ Vectorizer, job matrix and jobs saved!")

✅ Vectorizer, job matrix and jobs saved!
